# Lipsyncstudio on Kaggle (LatentSync 1.6 / 1.5, T4 ×2 or P100)

Runs the full `local_ui.py` dubbing studio and exposes it as a public `*.gradio.live` link.

This notebook builds a **Python 3.10** environment with [`uv`](https://github.com/astral-sh/uv), installs the project's *exact, proven* dependency lock, and downloads **both** LatentSync **1.6** (512px, default) and **1.5** (256px) — so either works in the UI with no missing-checkpoint error. Face detection, Whisper, and Demucs run on the **GPU** to keep the CPU free.

**Multi-GPU:** on `GPU T4 ×2` the app runs **one resident worker per T4** and renders a video's clips **in parallel across both cards** (~2× faster). It automatically uses fewer workers if host RAM/disk is tight (prevents truncated clips). Set **GPUs to use = 0** (all).

## Before you Run All
1. **Settings → Accelerator:** `GPU T4 ×2` (2×16 GB — recommended) or `GPU P100`. *Disabled? Phone-verify at kaggle.com/settings.*
2. **Settings → Internet:** `On` (same gate). Required for clone, uv, pip, and model downloads.
3. **Add-ons → Secrets:** add a secret named **`SIXTYDB_API_KEY`** with your 60db key and attach it.
4. **Run All.** First run ~14–20 min (env build + ~10 GB checkpoints for both versions). The last cell blocks and prints `Running on public URL: https://xxxx.gradio.live` — open it.

### Engine controls in the UI (the **Engine** section)
- **LatentSync version:** `1.6` (default, 512px) or `1.5` (256px) — both are downloaded.
- **GPU memory mode:** leave on `auto`.
- **GPUs to use:** `0` = every GPU.

The share link is public to anyone who has it and spends your 60db credits. It lives only while this session runs.

## 1. Clone the app and system libs

In [ ]:
import os, subprocess, pathlib

REPO = "https://github.com/RajEmaster/60db-lipsync-studio.git"
CLONE = "/kaggle/working/60db-lipsync-studio"
APP = CLONE + "/LatentSync-trial"

if not os.path.isdir(CLONE):
    subprocess.run(["git", "clone", "--depth", "1", REPO, CLONE], check=True)
else:
    subprocess.run(["git", "-C", CLONE, "pull", "--ff-only"], check=False)

subprocess.run("apt-get -qq update && apt-get -qq install -y libgl1 ffmpeg",
               shell=True, check=False)

os.chdir(APP)
print("App dir:", APP)

## 2. Build a Python 3.10 environment with uv
`uv` downloads a standalone CPython 3.10 and creates an isolated venv at `/kaggle/working/env310`.

In [ ]:
ENV = "/kaggle/working/env310"
PY = ENV + "/bin/python"

subprocess.run(["pip", "install", "-q", "uv"], check=True)
subprocess.run(["uv", "venv", "--python", "3.10", ENV], check=True)
print(subprocess.run([PY, "--version"], capture_output=True, text=True).stdout.strip())

## 3. Install the proven stack into the 3.10 env
torch 2.5.1+cu121 first, then the exact project lock (read from the repo) + the UI deps.

In [ ]:
# 3a. torch / torchvision / torchaudio from the cu121 index
subprocess.run([
    "uv", "pip", "install", "--python", PY,
    "torch==2.5.1", "torchvision==0.20.1", "torchaudio==2.5.1",
    "--index-url", "https://download.pytorch.org/whl/cu121",
], check=True)

In [ ]:

# 3b. Build requirements_kaggle.txt: pin ONLY model-critical pkgs, keep the rest unpinned
lock = pathlib.Path("trial-requirements-lock.txt").read_text(encoding="utf-16")
ui = pathlib.Path("requirements.txt").read_text(encoding="utf-8")
lines, have = [], set()
def name_of(s):
    return s.split("==")[0].split(">")[0].split("<")[0].strip().lower().replace("_", "-")
KEEP = {
    "numpy","diffusers","transformers","tokenizers","accelerate","omegaconf","einops",
    "decord","kornia","kornia-rs","opencv-python","onnx","onnxruntime","scikit-image",
    "scikit-learn","scipy","librosa","numba","llvmlite","soundfile","safetensors",
    "imageio","imageio-ffmpeg","ffmpeg-python","tiktoken","gradio",
    "realtimevad","kaldi-native-fbank","sixtydb","python-dotenv","psutil",
}
def emit(p):
    n = name_of(p)
    if n in have: return
    have.add(n); lines.append(p if n in KEEP else n)
for ln in lock.splitlines():
    p = ln.strip()
    if not p or p.startswith("#"): continue
    if p.lower().startswith(("torch==","torchvision==","torchaudio==")): continue
    emit(p)
for ln in ui.splitlines():
    p = ln.strip()
    if p and not p.startswith("#"): emit(p)
pathlib.Path("requirements_kaggle.txt").write_text("\n".join(lines) + "\n")
print("\n".join(lines))

In [ ]:
# 3c. Install the locked stack (~5-8 min)
subprocess.run(["uv", "pip", "install", "--python", PY,
                "-r", "requirements_kaggle.txt"], check=True)

In [ ]:
# 3d. Demucs for 'Restore original background' (optional).
# If this fails, the app still works - just leave that box unchecked in the UI.
subprocess.run(["uv", "pip", "install", "--python", PY, "demucs==4.0.1"], check=False)

## 4. Patch for Linux / this environment
The app spawns its GPU worker and the Demucs separator via `settings.WORKER_PYTHON`,
a hardcoded Windows `.venv` path. Repoint it at the 3.10 env's Python.

In [ ]:
s = pathlib.Path("settings.py")
txt = s.read_text()
old = "WORKER_PYTHON = BACKEND / '.venv' / 'Scripts' / 'python.exe'"
assert old in txt, "settings.py layout changed - inspect it manually"
s.write_text(txt.replace(old, f"WORKER_PYTHON = Path(r'{PY}')"))
print(s.read_text())

## 5. 60db credentials from the Kaggle secret

In [ ]:
from kaggle_secrets import UserSecretsClient

try:
    key = UserSecretsClient().get_secret("SIXTYDB_API_KEY").strip()
except Exception as exc:
    raise SystemExit(
        "Add a Kaggle secret named SIXTYDB_API_KEY (Add-ons -> Secrets) and attach it "
        "to this notebook, then Run All again. Original error: " + str(exc))

pathlib.Path(".env").write_text(
    f"SIXTYDB_API_KEY={key}\n"
    "SIXTYDB_SDK_BASE_URL=https://api.60db.ai\n"
    "SIXTYDB_TRANSLATION_MODEL=60db-tiny\n")
print("Wrote .env with SIXTYDB_API_KEY (len=%d)" % len(key))

## 6. Download checkpoints + face models

Downloads **both** versions so either works in the UI with no missing-checkpoint error: the shared VAE + Whisper + InsightFace models and the **1.5** UNet here, then the **1.6** UNet (512px) into `checkpoints/v1.6/` in the next cell. ~10 GB total.

In [ ]:
import sys
os.chdir(APP)
# Download BOTH versions so either works in the UI with no "checkpoint not found":
# the shared VAE + Whisper + the 1.5 UNet here, and the 1.6 UNet in the next cell.
subprocess.run([sys.executable, "download_trial.py"], check=True)
subprocess.run([sys.executable, "download_faces.py"], check=True)
subprocess.run("ls -lh checkpoints checkpoints/vae checkpoints/whisper "
               "checkpoints/auxiliary/models/buffalo_l", shell=True)

In [ ]:
# 6b. LatentSync 1.6 checkpoint (512px) -> checkpoints/v1.6/latentsync_unet.pt (~5 GB).
# REQUIRED for version 1.6 (the UI default). Without this the engine raises
# "LatentSync 1.6 checkpoint not found".
os.chdir(APP)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=False)
subprocess.run([sys.executable, "-c",
    "from huggingface_hub import hf_hub_download;"
    "p=hf_hub_download('ByteDance/LatentSync-1.6','latentsync_unet.pt',"
    "local_dir='checkpoints/v1.6');"
    "print('1.6 UNet ->', p)"], check=True)
subprocess.run("ls -lh checkpoints/v1.6", shell=True)

In [ ]:
# GPU sanity check inside the 3.10 env
print(subprocess.run(
    [PY, "-c", "import torch;print('CUDA', torch.cuda.is_available(),"
     "torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')"],
    capture_output=True, text=True).stdout)

## 7. Launch the UI (public share link)
Runs `local_ui.build_ui()` with `share=True` under the 3.10 env. This cell blocks and
keeps serving. Look for `Running on public URL: https://....gradio.live`.

In [ ]:
launcher = """import os
APP = os.path.dirname(os.path.abspath(__file__))
os.chdir(APP)
from local_ui import build_ui
build_ui().queue(default_concurrency_limit=1).launch(
    server_name="0.0.0.0", server_port=7860, share=True,
    allowed_paths=[os.path.join(APP, "trial_inputs"), os.path.join(APP, "local_jobs")],
)
"""
pathlib.Path(APP, "run_kaggle.py").write_text(launcher)
print("Launching... watch for the https://....gradio.live URL below.")

In [ ]:
import pathlib
shim = (
    'import fcntl\n'
    'LK_LOCK = "lock"; LK_RLCK = "rlck"; LK_NBLCK = "nblck"\n'
    'LK_NBRLCK = "nbrlck"; LK_UNLCK = "unlck"\n'
    'def locking(fd, mode, nbytes):\n'
    '    if mode == LK_UNLCK:\n'
    '        fcntl.flock(fd, fcntl.LOCK_UN)\n'
    '    elif mode in (LK_NBLCK, LK_NBRLCK):\n'
    '        fcntl.flock(fd, fcntl.LOCK_EX | fcntl.LOCK_NB)\n'
    '    else:\n'
    '        fcntl.flock(fd, fcntl.LOCK_EX)\n'
)
pathlib.Path(APP, "msvcrt.py").write_text(shim)
print("shim written ->", APP + "/msvcrt.py")

In [ ]:
!cd {APP} && MPLBACKEND=Agg {PY} run_kaggle.py